# 財務報表：資產負債表、損益表、現金流量表

從 MOPS 下載並解析 XBRL 財務報告。

In [1]:
from twmops import FinancialFetcher
import asyncio

## 報告類型

- `balance_sheet`: 財務狀況表
- `income_statement`: 綜合損益表
- `cash_flow`: 現金流量表
- `equity_statement`: 權益變動表

In [2]:
def fetch_balance_sheet():
    fetcher = FinancialFetcher()

    # 下載台積電 2024 年第 3 季資產負債表
    stmt = fetcher.get_financial_statement(
        stock_id="2330",
        year=113,
        quarter=3,
        report_type="balance_sheet",
        format="tree"  # 分層；使用 "flat" 取得清單
    )

    print(f"{stmt.stock_id} 資產負債表 - 第{stmt.quarter}季 民國{stmt.year}")
    print(f"幣別: {stmt.currency} | 單位: {stmt.unit}")
    print(f"總項目數: {len(stmt.items)}")
    print()

    # 列印頂層項目
    for item in stmt.items:
        print(f"{item.account_code} {item.account_name}: {item.value}")
        # 列印子項目（深度 1）
        for child in item.children[:3]:  # 顯示前 3 個子項目
            print(f"  └─ {child.account_code} {child.account_name}: {child.value}")
        if len(item.children) > 3:
            print(f"  ... 還有 {len(item.children) - 3} 個")

    return stmt

balance_sheet = fetch_balance_sheet()

Arelle not available — falling back to lxml parsing (no hierarchy/calculation)
SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi
No presentation arcs found, falling back to flat facts list


2330 Balance Sheet - Q3 113
Currency: TWD | Unit: thousands
Total items: 382

AccountantName AccountantName: None
AccountantsReportBody AccountantsReportBody: None
AccountsReceivableDuefromRelatedPartiesNet AccountsReceivableDuefromRelatedPartiesNet: 356257
AccountsReceivableNet AccountsReceivableNet: 222467653
AccumulatedInvestmentInMainlandChinaAtTheEndOfThePeriod AccumulatedInvestmentInMainlandChinaAtTheEndOfThePeriod: 49461079
AccumulatedInwardRemittanceOfEarningsAsOfTheEndOfThePeriod AccumulatedInwardRemittanceOfEarningsAsOfTheEndOfThePeriod: 0
AccumulatedOutflowOfInvestmentFromTaiwanAtTheBeginningOfThePeriod AccumulatedOutflowOfInvestmentFromTaiwanAtTheBeginningOfThePeriod: 30521412
AccumulatedOutflowOfInvestmentFromTaiwanAtTheEndOfThePeriod AccumulatedOutflowOfInvestmentFromTaiwanAtTheEndOfThePeriod: 30521412
AcquisitionOfFinancialAssetsAtAmortisedCost AcquisitionOfFinancialAssetsAtAmortisedCost: 115641029
AcquisitionOfFinancialAssetsAtFairValueThroughOtherComprehensiveIncome Ac

## 展平階層以進行分析

In [3]:
def fetch_flat_statement():
    fetcher = FinancialFetcher()

    # 同一份報表但展平
    stmt = fetcher.get_financial_statement(
        stock_id="2330",
        year=113,
        quarter=3,
        report_type="balance_sheet",
        format="flat"  # 所有項目同一層級
    )

    print(f"展平格式包含 {len(stmt.items)} 個項目（無階層）")
    print()

    # 搜尋特定科目
    cash = [item for item in stmt.items if '現金' in item.account_name]
    if cash:
        print(f"現金科目:")
        for item in cash:
            print(f"  {item.account_name}: {item.value}")

    return stmt

flat_stmt = fetch_flat_statement()

Arelle not available — falling back to lxml parsing (no hierarchy/calculation)
SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi
No presentation arcs found, falling back to flat facts list


Flat format has 382 items (no hierarchy)

Cash accounts:
  CashAndCashEquivalents: 1311806535
  CashAndCashEquivalentsAtBeginningOfPeriod: 1342814083
  CashAndCashEquivalentsAtEndOfPeriod: 1311806535
  CashDividendsOfOrdinaryShare: 226900972
  CashFlowsFromUsedInFinancingActivities: 129527119
  CashFlowsFromUsedInOperatingActivities: 847138000
  CashFlowsFromUsedInOperations: 969445435
  DerecognitionOfDerivativeFinancialLiabilitiesForHedging-CashFlowsFromUsedInInvestingActivities: 66776
  EffectOfExchangeRateChangesOnCashAndCashEquivalents: 25182665
  IncreaseDecreaseInCashAndCashEquivalents: 31007548
  NetCashFlowsFromUsedInInvestingActivities: 773801094
  OtherInflowsOutflowsOfCashClassifiedAsFinancingActivities: 27908580
  OtherInflowsOutflowsOfCashClassifiedAsInvestingActivities: 20757802


## 損益表

In [4]:
def fetch_income_statement():
    fetcher = FinancialFetcher()

    stmt = fetcher.get_financial_statement(
        stock_id="2330",
        year=113,
        quarter=3,
        report_type="income_statement",
        format="tree"
    )

    print(f"損益表 第{stmt.quarter}季 民國{stmt.year}")

    # 列印營收與利潤行
    for item in stmt.items[:10]:
        print(f"{item.account_name}: {item.value}")

    return stmt

income = fetch_income_statement()

Arelle not available — falling back to lxml parsing (no hierarchy/calculation)
SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi
No presentation arcs found, falling back to flat facts list


Income Statement Q3 113
AccountantName: None
AccountantsReportBody: None
AccountsReceivableDuefromRelatedPartiesNet: 356257
AccountsReceivableNet: 222467653
AccumulatedInvestmentInMainlandChinaAtTheEndOfThePeriod: 49461079
AccumulatedInwardRemittanceOfEarningsAsOfTheEndOfThePeriod: 0
AccumulatedOutflowOfInvestmentFromTaiwanAtTheBeginningOfThePeriod: 30521412
AccumulatedOutflowOfInvestmentFromTaiwanAtTheEndOfThePeriod: 30521412
AcquisitionOfFinancialAssetsAtAmortisedCost: 115641029
AcquisitionOfFinancialAssetsAtFairValueThroughOtherComprehensiveIncome: 54832622


## 簡化格式（FinMind 風格）

用於快速分析，不含樹狀階層。

In [5]:
def fetch_simplified():
    fetcher = FinancialFetcher()

    # 簡化格式，所有概念的平面清單
    stmt = fetcher.get_simplified_statement(
        stock_id="2330",
        year=113,
        quarter=3,
        statement_type="income_statement"
    )

    print(f"簡化損益表 ({stmt.statement_type})")
    print(f"報告日期: {stmt.report_date}")
    print(f"總項目數: {len(stmt.items)}")
    print()

    # 列印前 10 筆
    import pandas as pd
    df = pd.DataFrame([
        {'type': item.type, 'name': item.origin_name, 'value': item.value}
        for item in stmt.items[:10]
    ])
    print(df.to_string(index=False))

    return stmt

simplified = fetch_simplified()

Arelle not available — falling back to lxml parsing (no hierarchy/calculation)
SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi


Simplified Income Statement (income_statement)
Report date: 2024-09-30
Total items: 311

                                                            type                                                             name        value
                                                       CompanyID                                                        CompanyID       2330.0
                                                            Year                                                             Year       2024.0
                                                         Quarter                                                          Quarter          3.0
                                          CashAndCashEquivalents                                           CashAndCashEquivalents 1886780555.0
            CurrentFinancialAssetsAtFairValueThroughProfitOrLoss             CurrentFinancialAssetsAtFairValueThroughProfitOrLoss     971386.0
CurrentFinancialAssetsAtFairValueThroughOtherComprehe

## 註記

- 需要從 MOPS 下載 XBRL（首次 ~1-2 秒分類法快取，後續 <1 秒）
- 若未安裝 Arelle 則改用 lxml 解析（功能受限）
- 使用 Arelle 安裝：`pip install 'twmops[xbrl]'`